In [10]:
from dataclasses import dataclass
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()


@dataclass(frozen=True)
class Provider:
    """Configuration for an LLM provider."""
    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str


PROVIDERS = [
    Provider(
        "OpenAI",
        "OPENAI_API_KEY",
        True,
        None,
        "gpt-4o-mini",
    ),
    Provider(
        "Groq",
        "GROQ_API_KEY",
        True,
        "https://api.groq.com/openai/v1",
        "openai/gpt-oss-120b",
    ),
    Provider(
        "OpenRouter",
        "OPENROUTER_API_KEY",
        False,
        "https://openrouter.ai/api/v1",
        "openai/gpt-4o-mini",
    ),
]


def select_provider() -> Provider:
    # Explicit provider selection takes priority.
    selected = os.getenv("LLM_PROVIDER")

    if selected:
        for provider in PROVIDERS:
            if provider.name.lower() == selected.lower():
                if not os.getenv(provider.env_var):
                    raise RuntimeError(
                        f"{provider.env_var} is required "
                        f"for {provider.name}."
                    )
                return provider

        names = ", ".join(p.name for p in PROVIDERS)
        raise RuntimeError(
            f"Unknown provider: {selected}. Choose from: {names}"
        )

    # Otherwise, use the first configured provider.
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider

    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(
        f"No provider key found. Configure one of: {expected}"
    )


def build_client(provider: Provider) -> OpenAI:
    api_key = os.getenv(provider.env_var)

    if provider.base_url is None:
        return OpenAI(api_key=api_key)

    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url,
    )


def have_any_key() -> bool:
    return any(
        os.getenv(provider.env_var)
        for provider in PROVIDERS
    )


print(
    "Found a provider key."
    if have_any_key()
    else "No provider key found."
)

Found a provider key.


In [24]:
conversation = []


def send_message(user_input: str) -> str:
    """Append user input, get an LLM reply, and save both messages."""

    if not user_input.strip():
        return "Please enter a message."

    # Append the user's message
    conversation.append({
        "role": "user",
        "content": user_input,
    })

    try:
        # Send the complete conversation history
        reply = chat_reply(conversation)

        # Append the assistant's response
        conversation.append({
            "role": "assistant",
            "content": reply,
        })

        return reply

    except Exception:
        # Remove the user message if the request fails
        conversation.pop()
        raise

In [25]:
def llm_reply(prompt: str) -> str:
    """Send a single prompt to the selected LLM provider."""
    provider = select_provider()
    print(f"Using {provider.name} provider")

    client = build_client(provider)

    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=200,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    return result.choices[0].message.content or ""


def chat_reply(messages: list[dict]) -> str:
    """Send a conversation history to the selected LLM provider."""
    provider = select_provider()
    print(f"Using {provider.name} provider")

    client = build_client(provider)

    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=1000,
        messages=messages,
    )

    return result.choices[0].message.content or ""

In [12]:
prompt = "Explain cosine similarity in simple words."

try:
    print(llm_reply(prompt))
except Exception as e:
    print(f"Error: {e}")

Using OpenRouter provider
Error: Error code: 401 - {'error': {'message': 'Missing Authentication header', 'code': 401}}


In [21]:
import os
import requests
from dotenv import load_dotenv

load_dotenv(override=True)

key = os.getenv("OPENROUTER_API_KEY", "").strip()

print("Key exists:", bool(key))
print("Key length:", len(key))
print("Starts with sk-or-:", key.startswith("sk-or-"))

response = requests.get(
    "https://openrouter.ai/api/v1/key",
    headers={
        "Authorization": f"Bearer {key}",
        "Accept": "application/json",
    },
    timeout=20,
)

print("Status:", response.status_code)
print("Response:", response.text[:300])

Key exists: True
Key length: 56
Starts with sk-or-: False
Status: 401
Response: {"error":{"message":"Missing Authentication header","code":401}}


In [22]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

key = os.getenv("OPENROUTER_API_KEY", "").strip()

print("Length:", len(key))
print("Prefix:", repr(key[:8]))
print("Contains placeholder text:", "your_" in key.lower() or "paste_" in key.lower())
print("Contains spaces:", any(c.isspace() for c in key))

Length: 56
Prefix: 'gsk_i1Pp'
Contains placeholder text: False
Contains spaces: False


In [23]:
from dotenv import load_dotenv
import os

load_dotenv(override=True)

print("Groq key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("OpenRouter key loaded:", bool(os.getenv("OPENROUTER_API_KEY")))
print(
    "Keys are different:",
    os.getenv("GROQ_API_KEY") != os.getenv("OPENROUTER_API_KEY")
)

Groq key loaded: True
OpenRouter key loaded: True
Keys are different: False
